In [ ]:
# Cell 1 — 공통 실행 환경 설정
# 이 Cell은 모든 tutorial Notebook의 첫 번째 Cell이며 여러 번 실행해도 안전합니다.

from pathlib import Path
import importlib.util
import os
import subprocess
import sys

# Google Drive의 project folder가 다르면 이 값만 수정합니다.
COLAB_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/DiffDodree"
)

# 필요하면 환경변수로 project root를 직접 지정할 수 있습니다.
PROJECT_ROOT_OVERRIDE = os.environ.get("GUGAKGEN_PROJECT_ROOT")


def find_local_project_root(start: Path) -> Path:
    """현재 위치에서 pyproject.toml을 기준으로 project root를 찾습니다."""
    candidates = [start.resolve(), *start.resolve().parents]

    for candidate in candidates:
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "project root를 찾지 못했습니다. "
        "GUGAKGEN_PROJECT_ROOT를 지정하거나 project folder에서 Notebook을 실행하세요."
    )


try:
    # 순수 local Python에는 상위 ``google`` namespace가 없을 수 있습니다.
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    # 이미 mount되어 있으면 기존 mount를 그대로 사용합니다.
    drive.mount("/content/drive", force_remount=False)

if PROJECT_ROOT_OVERRIDE:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
elif IN_COLAB:
    project_root = COLAB_PROJECT_ROOT.resolve()
else:
    project_root = find_local_project_root(Path.cwd())

if not project_root.exists():
    raise FileNotFoundError(
        f"project folder가 존재하지 않습니다: {project_root}\n"
        "COLAB_PROJECT_ROOT 또는 GUGAKGEN_PROJECT_ROOT를 확인하세요."
    )

bootstrap_script = project_root / "scripts" / "bootstrap_notebook.py"

if not bootstrap_script.exists():
    raise FileNotFoundError(
        f"Notebook bootstrap script를 찾지 못했습니다: {bootstrap_script}"
    )

environment_profile = "colab" if IN_COLAB else "local"

# 필요한 dependency만 확인하고 누락된 package만 설치합니다.
# Colab profile에서는 Colab이 제공한 CUDA용 PyTorch를 교체하지 않습니다.
subprocess.run(
    [
        sys.executable,
        str(bootstrap_script),
        "--project-root",
        str(project_root),
        "--profile",
        environment_profile,
        "--install-missing",
    ],
    check=True,
)

# 현재 kernel에서 src package를 즉시 import할 수 있게 합니다.
src_root = project_root / "src"

if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

os.environ["GUGAKGEN_PROJECT_ROOT"] = str(project_root)
os.chdir(project_root)

from gugakgen.device import resolve_device
from gugakgen.reproducibility import set_global_seed

runtime = resolve_device(preference="auto")
device = runtime.device

SEED = 2026
set_global_seed(SEED)

print("환경 설정 완료")
print(f"- 실행 환경: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"- Project root: {project_root}")
print(f"- Device: {device}")
print(f"- AMP enabled: {runtime.amp_enabled}")
print(f"- Seed: {SEED}")

# 03. Transformer training

이 Notebook은 양청도드리 한 악보 안에서 반복적으로 corruption task를 구성하여
Transformer denoiser를 학습합니다. `smoke`는 MacBook CPU/MPS에서 pipeline을 확인하는
짧은 실행이고, `full`은 Colab 또는 Linux CUDA 환경에서 연구용 Checkpoint를 만드는 실행입니다.

학습에는 pitch, duration, family reconstruction과 함께 Graph, meter, 환입 Loss function을
사용합니다. `full` 실행은 CUDA가 없으면 즉시 중단됩니다.

In [ ]:
# 연구자가 주로 수정하는 training option입니다.
from copy import deepcopy

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import display

from gugakgen.config import load_yaml

RUN_MODE = "smoke"  # "smoke" 또는 "full"
SMOKE_STEPS = 3
FULL_STEPS = 6000
BATCH_SIZE = 32

# 각 Masking mode의 합은 1.0이어야 합니다.
MASKING_PROBABILITIES = {
    "event": 0.25,
    "measure": 0.15,
    "span": 0.15,
    "structure": 0.15,
    "joint": 0.15,
    "hwanip_target": 0.15,
}

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE는 'smoke' 또는 'full'이어야 합니다.")
if abs(sum(MASKING_PROBABILITIES.values()) - 1.0) > 1e-9:
    raise ValueError("MASKING_PROBABILITIES의 합은 1.0이어야 합니다.")
if RUN_MODE == "full" and runtime.backend != "cuda":
    raise RuntimeError("full training에는 CUDA가 필요합니다. Colab GPU runtime을 선택하세요.")

print("RUN_MODE:", RUN_MODE)
print("Device:", runtime.as_dict())

In [ ]:
# 원본 YAML을 보존하고 tutorial 전용 runtime configuration을 만듭니다.
base_config_path = project_root / "configs/train_maskgit.yaml"
training_config = deepcopy(load_yaml(base_config_path))
training_config["training"]["smoke_steps"] = SMOKE_STEPS
training_config["training"]["steps"] = FULL_STEPS
training_config["training"]["batch_size"] = BATCH_SIZE
training_config["training"]["masking_modes"] = list(MASKING_PROBABILITIES)
training_config["training"]["masking_probabilities"] = list(MASKING_PROBABILITIES.values())

runtime_config_dir = project_root / "artifacts/runtime_configs"
runtime_config_dir.mkdir(parents=True, exist_ok=True)
runtime_config_path = runtime_config_dir / "tutorial_train_maskgit.yaml"
runtime_config_path.write_text(
    yaml.safe_dump(training_config, allow_unicode=True, sort_keys=False),
    encoding="utf-8",
)

display(
    pd.DataFrame(
        [{"Masking mode": key, "probability": value} for key, value in MASKING_PROBABILITIES.items()]
    )
)
print("Runtime configuration:", runtime_config_path.relative_to(project_root))

In [ ]:
# 같은 API로 smoke 또는 full training을 실행합니다.
from gugakgen.workflows.train import train_from_config

training_result = train_from_config(
    runtime_config_path,
    project_root=project_root,
    run_mode=RUN_MODE,
    runtime=runtime,
)

print("Training 완료")
print("Checkpoint:", training_result.checkpoint_path)
print("SHA-256:", training_result.checkpoint_sha256)
print("History:", training_result.history_path)

In [ ]:
# Loss function history를 table과 curve로 확인합니다.
history = pd.DataFrame(training_result.history)
display(history)

loss_columns = [
    column
    for column in history.columns
    if column.startswith("eval_") and column not in {"eval_total"}
]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
history.plot(x="step", y=["train_total", "eval_total"], marker="o", ax=axes[0])
axes[0].set_title("Total Loss function")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.2)

history.plot(x="step", y=loss_columns, marker="o", ax=axes[1])
axes[1].set_title("Evaluation Loss function components")
axes[1].set_ylabel("Loss")
axes[1].grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 실행 결과의 용도

`*_smoke.pt`는 code path와 Device 호환성을 확인하기 위한 Checkpoint이며 논문 결과로
사용하지 않습니다. `full` 결과만 연구 실험에 사용하고, Checkpoint SHA-256과 runtime
metadata를 함께 보존합니다. MPS 결과와 CUDA 결과가 완전히 같은 floating-point 값을
가질 필요는 없지만, 동일 Seed의 실험 조건과 artifact provenance는 유지해야 합니다.